In [ ]:
#cell 1:
# Standard Python libraries
import builtins
import gc
import os
import re
import time

# Data science libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# PySpark core
import pyspark
from pyspark import RDD, SparkContext
from pyspark.sql import SparkSession

# PySpark SQL
from pyspark.sql.functions import (
    array, array_distinct, array_join, avg, col, explode,length, collect_list, concat, substring, count, date_trunc,
    expr, from_unixtime, lit, lower, regexp_replace, month, to_timestamp, udf, unix_timestamp, when, year,
    min as spark_min, max as spark_max
)
from pyspark.sql.types import ArrayType, DoubleType, FloatType, IntegerType, StringType

# PySpark ML
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression, LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.feature import (
    HashingTF, IDF, RegexTokenizer, SQLTransformer, StopWordsRemover, Tokenizer, VectorAssembler,NGram,CountVectorizer
)
from pyspark.ml.fpm import FPGrowth

# Other libraries
from dateutil.relativedelta import relativedelta
from datetime import datetime
from itertools import combinations
from tabulate import tabulate
from tenacity import retry, stop_after_attempt, wait_exponential
from textblob import TextBlob

# List files in /kaggle/input
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Function to check if SparkContext is active
def is_spark_context_active():
    try:
        SparkContext.getOrCreate().getConf().getAll()
        return True
    except Exception:
        return False

In [ ]:
#cell 2
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

In [ ]:
# # #cell 3:
# !ls /kaggle/working

In [ ]:
# Cell 4: Spark Configuration for 563,000 Rows on Kaggle
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Sentiment Analysis") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setCheckpointDir("/kaggle/working/checkpoints")

In [ ]:
#cell 5:
# Load dataset
# file_path = "/kaggle/input/amazon/Reviews.csv"
file_path = "/kaggle/input/original/Reviews.csv"
data = spark.read.csv(file_path, header=True, inferSchema=True)

In [ ]:
#cell 6:
data.first()

In [ ]:
#cell 7:
# data = data.withColumn("Time", to_timestamp(F.col("Time").cast("double")))

stats =data.withColumn("Time", to_timestamp(col("Time").cast("double"))).select(
    spark_min("Time").alias("min_time"),
    spark_max("Time").alias("max_time"),
    count("*").alias("row_count")
).first()

print(f"Minimum Time: {stats.min_time}")
print(f"Maximum Time: {stats.max_time}")
print(f"Number of Rows: {stats.row_count}")

In [ ]:
#cell 8:
# Function to display null counts and visualize them
def display_nulls(data):
    """
    Display null counts for each column and visualize them in a bar chart.

    Args:
        data (DataFrame): Spark DataFrame to analyze.
    """
    null_counts = data.select([count(when(col(c).isNull(), c)).alias(c) for c in data.columns])
    null_counts_pd = null_counts.toPandas()

    print("NULL counts per columns:")
    null_counts.show()

    plt.figure(figsize=(15, 8))
    plt.bar(null_counts_pd.columns, null_counts_pd.iloc[0])
    plt.title('NULL Values per Column')
    plt.xlabel('Columns')
    plt.ylabel('Number of NULL values')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

display_nulls(data)

In [ ]:
#cell 9:
data.columns

In [ ]:
#cell 10:
class ReviewAnalyzer:
    def __init__(self, spark_session):
        self.spark = spark_session
        self.preprocessed_data = None
        self.historical_data = None

    def preprocess(self, input_data):
        if isinstance(input_data, str):
            raw_data = self.spark.read.csv(input_data, header=True, inferSchema=True)
        else:
            raw_data = input_data

        # Validate input data
        if raw_data is None:
            raise ValueError("Input data is invalid!")

        valid_data = raw_data.filter(
            (col("Time").isNotNull()) &
            (col("Time") > 0) &
            (col("Time") < int(time.time()))
        )
        filtered_data = valid_data.na.drop(subset=[
            "HelpfulnessNumerator", "HelpfulnessDenominator",
            "Score", "Time", "Summary", "Text"
        ])

        preprocessing_stages = [
            SQLTransformer(statement="""SELECT
                Id, ProductId, UserId, Score,
                ROUND(((HelpfulnessNumerator + 1) * 100.0) / (HelpfulnessDenominator + 1), 2) as HelpfulnessScore,
                from_unixtime(Time) as ReviewDateTime, Time as RawTimestamp,
                Summary, Text
            FROM __THIS__""")
        ]

        pipeline = Pipeline(stages=preprocessing_stages)
        self.preprocessed_data = (
            pipeline.fit(filtered_data)
            .transform(filtered_data)
            .withColumn("Year", year(col("ReviewDateTime")))
            .withColumn("Month", month(col("ReviewDateTime")))
            .filter(col("Year").between(2000, 2022))
        )

        removed_count = raw_data.count() - self.preprocessed_data.count()
        print(f"Removed {removed_count} invalid entries during preprocessing")
        return self

    def load_incremental_batch(self, file_path, start_time, end_time):
        """Load data for a specific time window"""
        raw_data = self.spark.read.csv(file_path, header=True, inferSchema=True)
        filtered_data = raw_data.filter(
            (col("Time") >= start_time) &
            (col("Time") < end_time)
        )
        self.preprocess(filtered_data)  # Process and set self.preprocessed_data
        return self  # Return the instance, not the DataFrame

    def update_historical_data(self, new_batch):
        """Merge new batch with historical data"""
        if self.historical_data:
            self.historical_data = self.historical_data.union(new_batch)
        else:
            self.historical_data = new_batch

In [ ]:
#cell 11:
# Class for data visualizations
class DataAnalyzer:
    def __init__(self, spark_session, preprocessed_data):
        self.spark = spark_session
        self.data = preprocessed_data

    def generate_visualizations(self):
        df = self.data.toPandas()
        plt.figure(figsize=(20, 15))
        plt.subplot(2, 2, 1)
        df.groupby('Year')['Id'].count().plot(kind='bar')
        plt.title('Review Count by Year')
        plt.xlabel('Year')
        plt.ylabel('Number of Reviews')
        plt.xticks(rotation=45)
        plt.subplot(2, 2, 2)
        df['Score'].hist(bins=20)
        plt.title('Review Score Distribution')
        plt.xlabel('Score')
        plt.ylabel('Frequency')
        plt.subplot(2, 2, 3)
        helpfulness_by_year = df.groupby('Year')['HelpfulnessScore'].mean()
        helpfulness_by_year.plot(kind='line', marker='o')
        plt.title('Average Helpfulness Score by Year')
        plt.xlabel('Year')
        plt.ylabel('Avg Helpfulness Score')
        plt.subplot(2, 2, 4)
        correlation_cols = ['Score', 'HelpfulnessScore', 'Year']
        correlation_matrix = df[correlation_cols].corr()
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
        plt.title('Correlation Heatmap')
        plt.tight_layout()
        plt.savefig('/kaggle/working/data_analysis_plots.png')

    def statistical_summary(self):
        summary = {
            'total_reviews': self.data.count(),
            'unique_products': self.data.select('ProductId').distinct().count(),
            'unique_users': self.data.select('UserId').distinct().count(),
            'year_range': self.data.select(
                spark_min("Year").alias("min_year"),
                spark_max("Year").alias("max_year")
            ).first().asDict(),
            'score_stats': self.data.select('Score').describe().first().asDict(),
            'helpfulness_stats': self.data.select('HelpfulnessScore').describe().first().asDict()
        }
        print("Data Overview:")
        for key, value in summary.items():
            print(f"{key}: {value}")
        return summary

    def visualize_results(product_sentiments_df):
        """
        Create visualizations for sentiment analysis results

        Args:
            product_sentiments_df: Spark DataFrame with product sentiments
        """
        try:
            # Convert to Pandas for visualization
            pd_df = product_sentiments_df.toPandas()

            plt.figure(figsize=(15, 5))

            # Sentiment distribution
            plt.subplot(131)
            sentiment_dist = pd_df[['positive_ratio', 'neutral_ratio', 'negative_ratio']].mean()
            sentiment_dist.plot(kind='bar')
            plt.title('Overall Sentiment Distribution')
            plt.ylabel('Ratio')

            # Review count distribution
            plt.subplot(132)
            pd_df['review_count'].hist(bins=50)
            plt.title('Review Count Distribution')
            plt.xlabel('Number of Reviews')
            plt.ylabel('Frequency')

            # Products by dominant sentiment
            plt.subplot(133)
            dominant_sentiment = pd_df[['positive_ratio', 'neutral_ratio', 'negative_ratio']].idxmax(axis=1)
            dominant_sentiment.value_counts().plot(kind='pie', autopct='%1.1f%%')
            plt.title('Products by Dominant Sentiment')

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"Error in visualization: {str(e)}")

In [ ]:
#cell 12:
class SentimentAnalyzer:
    def __init__(self, spark_session):
        self.spark = spark_session
        self.sentiment_model_phase1 = None
        self.sentiment_model_phase2 = None
        self.batch_metrics = []
        self.evaluator = MulticlassClassificationEvaluator()

    def preprocess_text(self, df):
        """
        Enhanced preprocessing with better error handling and explicit column processing
        """
        try:
            # First, handle null values in text columns
            print("Checking for null values in text columns...")
            null_summary_count = df.filter(col("Summary").isNull()).count()
            null_text_count = df.filter(col("Text").isNull()).count()
            print(f"Found {null_summary_count} null values in Summary and {null_text_count} null values in Text")

            # Fill null values with empty strings before casting
            df = df.na.fill({"Summary": "", "Text": ""})

            # Ensure text columns are properly cast to string
            df = df.withColumn("Summary", when(col("Summary").isNull(), "").otherwise(col("Summary").cast("string")))
            df = df.withColumn("Text", when(col("Text").isNull(), "").otherwise(col("Text").cast("string")))

            # Register UDFs with proper return types and null handling
            def _is_subjective_summary(text):
                if not text or not isinstance(text, str) or len(text.strip()) == 0:
                    return 0
                try:
                    subjective_score = TextBlob(text).sentiment.subjectivity
                    return 1 if subjective_score > 0.5 else 0
                except:
                    return 0

            def _sentiment_score(text):
                if not text or not isinstance(text, str) or len(text.strip()) == 0:
                    return 0.0
                try:
                    return TextBlob(text).sentiment.polarity
                except:
                    return 0.0

            # Register UDFs with explicit return types
            subjective_udf = udf(_is_subjective_summary, IntegerType())
            sentiment_udf = udf(_sentiment_score, DoubleType())

            # Calculate sentiment scores
            result_df = (df
                .withColumn("is_subjective", subjective_udf(col("Summary")))
                .withColumn("sentiment_score", sentiment_udf(col("Text")))
                .withColumn("sentiment_polarity",
                    when(col("sentiment_score") > 0, "Positive")
                    .when(col("sentiment_score") < 0, "Negative")
                    .otherwise("Neutral"))
            )

            # Calculate class weights
            sentiment_counts = result_df.groupBy("sentiment_polarity").count()
            total = result_df.count()

            # With explicit casting and null handling
            weights = {row["sentiment_polarity"]: float(total) / (3.0 * float(row["count"]))
                      for row in sentiment_counts.collect() if row["sentiment_polarity"]}

            # Apply weights with null handling
            result_df = (result_df
                .withColumn("class_weight",
                    when(col("sentiment_polarity") == "Positive", lit(weights.get("Positive", 1.0)))
                    .when(col("sentiment_polarity") == "Negative", lit(weights.get("Negative", 1.0)))
                    .otherwise(lit(weights.get("Neutral", 1.0))))
                .cache())

            print(f"Processed {result_df.count()} reviews with sentiment analysis")
            return result_df

        except Exception as e:
            print(f"Error in preprocess_text: {str(e)}")
            import traceback
            traceback.print_exc()
            return None

    def aggregate_predictions(self, predictions_df):
        """
        Aggregate predictions by product with proper error handling

        Args:
            predictions_df: Spark DataFrame with predictions
        Returns:
            DataFrame with aggregated product sentiments
        """
        try:
            # Map numeric predictions back to sentiment labels
            sentiment_mapping = (
                when(col("phase2_prediction") == 2, "Positive")
                .when(col("phase2_prediction") == 1, "Neutral")
                .when(col("phase2_prediction") == 0, "Negative")
                .otherwise("Unknown")
            )

            predictions_with_labels = predictions_df.withColumn(
                "predicted_sentiment",
                sentiment_mapping
            )

            return (predictions_with_labels
                .groupBy("ProductId")
                .agg(
                    collect_list("predicted_sentiment").alias("sentiments"),
                    count("*").alias("review_count"),
                    avg(when(col("predicted_sentiment") == "Positive", 1).otherwise(0))
                        .alias("positive_ratio"),
                    avg(when(col("predicted_sentiment") == "Negative", 1).otherwise(0))
                        .alias("negative_ratio"),
                    avg(when(col("predicted_sentiment") == "Neutral", 1).otherwise(0))
                        .alias("neutral_ratio")
                )
                .withColumn("dominant_sentiment",
                    when(col("positive_ratio") > col("negative_ratio") and
                         col("positive_ratio") > col("neutral_ratio"), "Positive")
                    .when(col("negative_ratio") > col("positive_ratio") and
                          col("negative_ratio") > col("neutral_ratio"), "Negative")
                    .otherwise("Neutral"))
                .cache()
            )

        except Exception as e:
            print(f"Error in prediction aggregation: {str(e)}")
            return None

    def get_model_insights(self):
        """
        New method to provide model insights
        """
        if not self.batch_metrics:
            return "No metrics available"

        insights = {
            'phase1': {
                'avg_accuracy': np.mean([m['accuracy'] for m in self.batch_metrics if m['phase'] == 1]),
                'avg_f1': np.mean([m['f1'] for m in self.batch_metrics if m['phase'] == 1]),
                'subjective_ratio_trend': [m['subjective_ratio'] for m in self.batch_metrics if m['phase'] == 1]
            },
            'phase2': {
                'avg_accuracy': np.mean([m['accuracy'] for m in self.batch_metrics if m['phase'] == 2]),
                'avg_f1': np.mean([m['f1'] for m in self.batch_metrics if m['phase'] == 2]),
                'class_distribution_trend': [m['class_distribution'] for m in self.batch_metrics if m['phase'] == 2]
            }
        }
        return insights

        insights = {
            'phase1': {
                'avg_accuracy': np.mean([m['accuracy'] for m in self.batch_metrics if m['phase'] == 1]),
                'avg_f1': np.mean([m['f1'] for m in self.batch_metrics if m['phase'] == 1]),
                'subjective_ratio_trend': [m['subjective_ratio'] for m in self.batch_metrics if m['phase'] == 1]
            },
            'phase2': {
                'avg_accuracy': np.mean([m['accuracy'] for m in self.batch_metrics if m['phase'] == 2]),
                'avg_f1': np.mean([m['f1'] for m in self.batch_metrics if m['phase'] == 2]),
                'class_distribution_trend': [m['class_distribution'] for m in self.batch_metrics if m['phase'] == 2]
            }
        }
        return insights


    def train_phase1_classifier(self, training_data, batch_id=None):
        """Enhanced Phase 1 training with metrics and null handling"""
        try:
            # Explicitly check and handle nulls in the Summary column
            null_count = training_data.filter(col("Summary").isNull()).count()
            if null_count > 0:
                print(f"Found {null_count} null values in Summary column. Filling with empty strings.")
                training_data = training_data.na.fill({"Summary": ""})

            # Check empty strings as well
            empty_count = training_data.filter(col("Summary") == "").count()
            if empty_count > 0:
                print(f"Found {empty_count} empty strings in Summary column.")

            # Ensure all values are properly cast to strings
            training_data = training_data.withColumn("Summary",
                                                  when(col("Summary").isNull(), "")
                                                  .otherwise(col("Summary").cast("string")))

            # Use RegexTokenizer which handles nulls better than standard Tokenizer
            tokenizer = RegexTokenizer(inputCol="Summary", outputCol="phase1_words", pattern="\\W")
            remover = StopWordsRemover(inputCol="phase1_words", outputCol="phase1_filtered_words")

            # Add a length check to avoid empty arrays which can cause problems in HashingTF
            add_length = udf(lambda arr: [x for x in arr if x], ArrayType(StringType()))
            pipeline_stages = [
                tokenizer,
                remover,
                SQLTransformer(statement="""
                    SELECT *,
                    CASE WHEN size(phase1_filtered_words) = 0 THEN array('placeholder')
                    ELSE phase1_filtered_words END AS phase1_filtered_words_safe
                    FROM __THIS__
                """)
            ]

            # Continue with the rest of your pipeline
            hashingTF = HashingTF(inputCol="phase1_filtered_words_safe", outputCol="phase1_raw_features", numFeatures=1000)
            idf = IDF(inputCol="phase1_raw_features", outputCol="phase1_features", minDocFreq=2)

            phase1_lr = LogisticRegression(
                featuresCol="phase1_features",
                labelCol="is_subjective",
                predictionCol="phase1_prediction",
                maxIter=20,
                regParam=0.01
            )

            pipeline = Pipeline(stages=pipeline_stages + [hashingTF, idf, phase1_lr])

            # Split data for evaluation
            train_data, eval_data = training_data.randomSplit([0.8, 0.2])
            self.sentiment_model_phase1 = pipeline.fit(train_data)

            # Evaluate on held-out data
            predictions = self.sentiment_model_phase1.transform(eval_data)
            metrics = self.evaluate_predictions(predictions, phase=1, batch_id=batch_id)

            print(f"\nPhase 1 Training Metrics (Batch {batch_id}):")
            print(f"Accuracy: {metrics['accuracy']:.3f}")
            print(f"Precision: {metrics['precision']:.3f}")
            print(f"Recall: {metrics['recall']:.3f}")
            print(f"F1 Score: {metrics['f1']:.3f}")
            print(f"Subjective Ratio: {metrics['subjective_ratio']:.3f}")

            return metrics
        except Exception as e:
            print(f"Error in train_phase1_classifier: {str(e)}")
            import traceback
            traceback.print_exc()
            raise

    def train_phase2_classifier(self, training_data, batch_id=None):
        """Enhanced Phase 2 training with better class balancing"""
        # Prepare data
        label_mapping = when(col("sentiment_polarity") == "Positive", 2) \
            .when(col("sentiment_polarity") == "Neutral", 1) \
            .otherwise(0)
        training_data = training_data.withColumn(
            "sentiment_numeric_label",
            label_mapping.cast(IntegerType())
        )

        # Calculate and adjust class weights more aggressively for better balance
        class_counts = training_data.groupBy("sentiment_numeric_label").count().toPandas()
        total = class_counts['count'].sum()
        max_count = class_counts['count'].max()

        # Inverse frequency weighting with additional scaling for minority classes
        class_weights = {
            row['sentiment_numeric_label']: (max_count / row['count']) * (
                3.0 if row['sentiment_numeric_label'] == 0 else  # More weight to negative class
                0.2 if row['sentiment_numeric_label'] == 1 else  # Moderate weight to neutral class
                14.5  # Reduce weight of majority positive class
            )
            for _, row in class_counts.iterrows()
        }

        # Add class weights to training data
        training_data = training_data.withColumn(
            "class_weight",
            when(col("sentiment_numeric_label") == 0, lit(class_weights[0]))
            .when(col("sentiment_numeric_label") == 1, lit(class_weights[1]))
            .when(col("sentiment_numeric_label") == 2, lit(class_weights[2]))
        )

        # Undersample majority class and oversample minority classes
        def balance_classes(df):
            counts = df.groupBy("sentiment_numeric_label").count().collect()
            min_count = min(row['count'] for row in counts)
            balanced_dfs = []

            for row in counts:
                label = row['sentiment_numeric_label']
                count = row['count']

                if label == 2:  # Majority class (Positive)
                    # Undersample to 50% more than minimum class
                    sample_ratio = (min_count * 1.5) / count
                    label_df = df.filter(col("sentiment_numeric_label") == label).sample(False, sample_ratio)
                else:  # Minority classes
                    # Oversample to 75% of the reduced majority class size
                    sample_ratio = min(2.0, (min_count * 1.5 * 0.75) / count)
                    label_df = df.filter(col("sentiment_numeric_label") == label).sample(True, sample_ratio)

                balanced_dfs.append(label_df)

            return balanced_dfs[0].union(balanced_dfs[1]).union(balanced_dfs[2])

        # Apply balancing
        balanced_training_data = balance_classes(training_data)

        # Create and train pipeline with adjusted parameters
        tokenizer = Tokenizer(inputCol="Text", outputCol="phase2_words")
        remover = StopWordsRemover(inputCol="phase2_words", outputCol="phase2_filtered_words")
        hashingTF = HashingTF(inputCol="phase2_filtered_words", outputCol="phase2_raw_features", numFeatures=500)  # Increased features
        idf = IDF(inputCol="phase2_raw_features", outputCol="phase2_features")

        phase2_lr = LogisticRegression(
            featuresCol="phase2_features",
            labelCol="sentiment_numeric_label",
            predictionCol="phase2_prediction",
            weightCol="class_weight",
            maxIter=100,  # Increased iterations
            regParam=0.1,  # Added regularization
            elasticNetParam=0.5  # Added elasticNet mixing
        )

        pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, phase2_lr])

        # Split data and train with stratification
        train_data, eval_data = balanced_training_data.randomSplit([0.8, 0.2], seed=42)
        self.sentiment_model_phase2 = pipeline.fit(train_data)

        # Evaluate
        predictions = self.sentiment_model_phase2.transform(eval_data)
        metrics = self.evaluate_predictions(predictions, phase=2, batch_id=batch_id)

        # Print detailed metrics
        print(f"\nPhase 2 Training Metrics (Batch {batch_id}):")
        print(f"Accuracy: {metrics['accuracy']:.3f}")
        print(f"Precision: {metrics['precision']:.3f}")
        print(f"Recall: {metrics['recall']:.3f}")
        print(f"F1 Score: {metrics['f1']:.3f}")
        print("\nClass Distribution after balancing:")
        for label, pct in metrics['class_distribution'].items():
            print(f"Class {label}: {pct:.3f}")

        return metrics

    def visualize_batch_metrics(self):
        """Visualize metrics across all processed batches"""
        if not self.batch_metrics:
            print("No batch metrics available")
            return

    def predict(self, data):
        """
        Generate predictions using both phase 1 and phase 2 models with column conflict handling
        """
        if not self.sentiment_model_phase1 or not self.sentiment_model_phase2:
            raise ValueError("Models must be trained before making predictions")

        try:
            # Drop any existing prediction columns to avoid conflicts
            columns_to_drop = ['rawPrediction', 'probability', 'phase1_prediction', 'phase2_prediction']
            clean_data = data
            for col_name in columns_to_drop:
                if col_name in data.columns:
                    clean_data = clean_data.drop(col_name)

            # Generate Phase 1 predictions
            phase1_predictions = self.sentiment_model_phase1.transform(clean_data)

            # Only process subjective reviews in Phase 2
            subjective_reviews = phase1_predictions.filter(col("phase1_prediction") == 1)

            if subjective_reviews.count() > 0:
                # Clean prediction columns again before phase 2
                for col_name in columns_to_drop:
                    if col_name in subjective_reviews.columns:
                        subjective_reviews = subjective_reviews.drop(col_name)

                # Add sentiment numeric label for evaluation
                label_mapping = when(col("sentiment_polarity") == "Positive", 2) \
                    .when(col("sentiment_polarity") == "Neutral", 1) \
                    .otherwise(0)

                subjective_reviews = subjective_reviews.withColumn(
                    "sentiment_numeric_label",
                    label_mapping.cast(IntegerType())
                )

                # Generate Phase 2 predictions
                phase2_predictions = self.sentiment_model_phase2.transform(subjective_reviews)

                # Combine predictions
                final_predictions = phase1_predictions.join(
                    phase2_predictions.select("Id", "phase2_prediction"),
                    on="Id",
                    how="left"
                )

                # Fill in Neutral sentiment for objective reviews
                final_predictions = final_predictions.withColumn(
                    "phase2_prediction",
                    when(col("phase1_prediction") == 0, 1).otherwise(col("phase2_prediction"))
                )

                # Add predicted_sentiment column
                final_predictions = final_predictions.withColumn(
                    "predicted_sentiment",
                    when(col("phase2_prediction") == 2, "Positive")
                    .when(col("phase2_prediction") == 1, "Neutral")
                    .otherwise("Negative")
                )

                return final_predictions
            else:
                return phase1_predictions.withColumn("phase2_prediction", lit(1)).withColumn(
                    "predicted_sentiment", lit("Neutral")  # Since all are objective, they are Neutral by default
                )

        except Exception as e:
            print(f"Error in prediction: {str(e)}")
            return None

    def aggregate_predictions(predictions_df):
        """
        Aggregate predictions by product with proper error handling

        Args:
            predictions_df: Spark DataFrame with predictions
        Returns:
            DataFrame with aggregated product sentiments
        """
        try:
            return (
                predictions_df
                .groupBy("ProductId")
                .agg(
                    collect_list("sentiment_polarity").alias("sentiments"),
                    count("*").alias("review_count"),
                    avg(when(col("sentiment_polarity") == "Positive", 1).otherwise(0)).alias("positive_ratio"),
                    avg(when(col("sentiment_polarity") == "Negative", 1).otherwise(0)).alias("negative_ratio"),
                    avg(when(col("sentiment_polarity") == "Neutral", 1).otherwise(0)).alias("neutral_ratio")
                )
                .cache()
            )
        except Exception as e:
            print(f"Error in prediction aggregation: {str(e)}")
            return None


    def evaluate_predictions(self, predictions_df, phase=1, batch_id=None):
        """
        Evaluate model predictions and return metrics

        Args:
            predictions_df: DataFrame with predictions
            phase: 1 for subjectivity detection, 2 for sentiment analysis
            batch_id: Optional identifier for the batch

        Returns:
            Dictionary containing evaluation metrics
        """
        try:
            if phase == 1:
                # Evaluate Phase 1 (subjectivity detection)
                metrics = {
                    'accuracy': self.evaluator.setMetricName("accuracy")
                        .setLabelCol("is_subjective")
                        .setPredictionCol("phase1_prediction")
                        .evaluate(predictions_df),
                    'precision': self.evaluator.setMetricName("weightedPrecision")
                        .setLabelCol("is_subjective")
                        .setPredictionCol("phase1_prediction")
                        .evaluate(predictions_df),
                    'recall': self.evaluator.setMetricName("weightedRecall")
                        .setLabelCol("is_subjective")
                        .setPredictionCol("phase1_prediction")
                        .evaluate(predictions_df),
                    'f1': self.evaluator.setMetricName("f1")
                        .setLabelCol("is_subjective")
                        .setPredictionCol("phase1_prediction")
                        .evaluate(predictions_df),
                    'phase': 1,
                    'batch_id': batch_id,
                    'subjective_ratio': (predictions_df.filter(col("phase1_prediction") == 1)
                        .count() / predictions_df.count())
                }
            else:
                # Evaluate Phase 2 (sentiment analysis)
                metrics = {
                    'accuracy': self.evaluator.setMetricName("accuracy")
                        .setLabelCol("sentiment_numeric_label")
                        .setPredictionCol("phase2_prediction")
                        .evaluate(predictions_df),
                    'precision': self.evaluator.setMetricName("weightedPrecision")
                        .setLabelCol("sentiment_numeric_label")
                        .setPredictionCol("phase2_prediction")
                        .evaluate(predictions_df),
                    'recall': self.evaluator.setMetricName("weightedRecall")
                        .setLabelCol("sentiment_numeric_label")
                        .setPredictionCol("phase2_prediction")
                        .evaluate(predictions_df),
                    'f1': self.evaluator.setMetricName("f1")
                        .setLabelCol("sentiment_numeric_label")
                        .setPredictionCol("phase2_prediction")
                        .evaluate(predictions_df),
                    'phase': 2,
                    'batch_id': batch_id
                }

                # Add class distribution for Phase 2
                class_counts = predictions_df.groupBy("sentiment_numeric_label").count().collect()
                total = predictions_df.count()
                metrics['class_distribution'] = {
                    row['sentiment_numeric_label']: row['count'] / total
                    for row in class_counts
                }

            # Store metrics for later analysis
            self.batch_metrics.append(metrics)
            return metrics

        except Exception as e:
            print(f"Error in evaluation: {str(e)}")
            return {
                'accuracy': 0.0,
                'precision': 0.0,
                'recall': 0.0,
                'f1': 0.0,
                'phase': phase,
                'batch_id': batch_id,
                'error': str(e)
            }

In [ ]:
#cell 13:
# Main execution for preprocessing
print("Starting Sentiment Analysis...")
analyzer = ReviewAnalyzer(spark).preprocess(file_path)
preprocessed_data = analyzer.preprocessed_data

# After preprocessing, use DataAnalyzer
data_analyzer = DataAnalyzer(spark, preprocessed_data)

# Generate visualizations
data_analyzer.generate_visualizations()

# Get statistical summary
summary = data_analyzer.statistical_summary()

sentiment_analyzer = SentimentAnalyzer(spark)
sentiment_data = sentiment_analyzer.preprocess_text(preprocessed_data)

In [ ]:
#cell 14:
# Get min and max timestamps from the preprocessed data
min_max = preprocessed_data.select(
    spark_min(col("RawTimestamp").cast("long")).alias("min_ts"),
    spark_max(col("RawTimestamp").cast("long")).alias("max_ts")
).first()

min_ts = min_max["min_ts"]
max_ts = min_max["max_ts"]

# Convert Unix timestamps to datetime objects
if min_ts is not None and max_ts is not None:
    start_date = datetime.utcfromtimestamp(min_ts)
    end_date = datetime.utcfromtimestamp(max_ts)
    print(f"Start Date: {start_date}")
    print(f"End Date: {end_date}")
else:
    print("No valid timestamps found in the data.")

In [ ]:
# Cell 15: BatchProcessor Class Definition
class BatchProcessor:
    def __init__(self, spark_session, file_path, analyzer, sentiment_analyzer):
        self.spark = spark_session
        self.file_path = file_path
        self.analyzer = analyzer
        self.sentiment_analyzer = sentiment_analyzer
        self.all_predictions = None
        self.batch_metrics = []  # Changed from batch_stats to batch_metrics for consistency with SentimentAnalyzer
        self.min_batch_size = 1000
        self.accumulated_data = None

    def process_all_batches(self, time_windows):
        """
        Process all time windows sequentially

        Args:
            time_windows: List of (start_timestamp, end_timestamp) tuples
        Returns:
            DataFrame with final predictions
        """
        try:
            # Ensure Spark session is active

            # Process each time window
            for start_ts, end_ts in time_windows:
                print(f"Processing time window: {datetime.utcfromtimestamp(start_ts)} to {datetime.utcfromtimestamp(end_ts)}")

                # Process the batch for this time window
                batch_predictions = self.process_batch(start_ts, end_ts)

            # Return aggregated predictions
            return self.all_predictions

        except Exception as e:
            print(f"Error processing all batches: {str(e)}")
            import traceback
            traceback.print_exc()
            return None

    def accumulate_small_batch(self, batch_data):
        """Accumulate small batches until they reach minimum size"""
        if self.accumulated_data is None:
            self.accumulated_data = batch_data
        else:
            self.accumulated_data = self.accumulated_data.union(batch_data)

        count = self.accumulated_data.count()
        print(f"Accumulated batch size: {count}")
        return count >= self.min_batch_size

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=10))
    def process_batch(self, start_ts, end_ts):
        """Process a single batch with improved error handling"""
        try:
            batch_id = f"{datetime.utcfromtimestamp(start_ts).strftime('%Y%m')}"
            print(f"\nProcessing batch {batch_id}...")

            # Load and preprocess batch
            analyzer_instance = self.analyzer.load_incremental_batch(self.file_path, start_ts, end_ts)
            if not analyzer_instance or not analyzer_instance.preprocessed_data:
                return None

            batch_data = analyzer_instance.preprocessed_data.persist()
            batch_count = batch_data.count()
            print(f"Loaded {batch_count} reviews")

            # Accumulate small batches
            if batch_count < self.min_batch_size:
                if not self.accumulate_small_batch(batch_data):
                    batch_data.unpersist()
                    return None

            # Process batch
            data_to_process = self.accumulated_data if self.accumulated_data else batch_data
            sentiment_data = self.sentiment_analyzer.preprocess_text(data_to_process)
            if not sentiment_data:
                return None

            # Train and predict
            phase1_metrics = self.sentiment_analyzer.train_phase1_classifier(sentiment_data, batch_id)
            phase2_metrics = self.sentiment_analyzer.train_phase2_classifier(sentiment_data, batch_id)
            predictions = self.sentiment_analyzer.predict(sentiment_data)

            # Store predictions
            if predictions is not None:
                if self.all_predictions is None:
                    self.all_predictions = predictions
                else:
                    self.all_predictions = self.all_predictions.union(predictions)
                self.all_predictions = self.all_predictions.persist()

            # Store metrics
            self.batch_metrics.extend([
                {**phase1_metrics, 'phase': 1, 'batch_id': batch_id},
                {**phase2_metrics, 'phase': 2, 'batch_id': batch_id}
            ])

            # Reset accumulated data
            if self.accumulated_data:
                self.accumulated_data.unpersist()
                self.accumulated_data = None

            return predictions

        except Exception as e:
            print(f"Error processing batch {batch_id}: {str(e)}")
            import traceback
            traceback.print_exc()
            return None
        finally:
            if 'batch_data' in locals() and not batch_data.is_cached:
                batch_data.unpersist()
            gc.collect()

    def get_batch_statistics(self):
        """Return comprehensive summary statistics for all processed batches"""
        if not self.batch_metrics:
            return {
                'total_batches': 0,
                'total_reviews': 0,
                'avg_reviews_per_batch': 0,
                'performance_metrics': {
                    'accuracy': {'mean': 0, 'min': 0, 'max': 0},
                    'f1': {'mean': 0, 'min': 0, 'max': 0},
                    'precision': {'mean': 0, 'min': 0, 'max': 0},
                    'recall': {'mean': 0, 'min': 0, 'max': 0}
                },
                'sentiment_distribution': {
                    'positive': 0,
                    'neutral': 0,
                    'negative': 0
                }
            }

        # Calculate total reviews from predictions
        total_reviews = self.all_predictions.count() if self.all_predictions else 0

        # Calculate metrics
        metrics_by_phase = {1: [], 2: []}
        for metric in self.batch_metrics:
            metrics_by_phase[metric['phase']].append(metric)

        def calculate_metric_stats(metrics, key):
            values = [m[key] for m in metrics if key in m]
            if not values:
                return {'mean': 0, 'min': 0, 'max': 0}
            return {
                'mean': builtins.sum(values) / len(values),
                'min': min(values),
                'max': max(values)
            }

        phase2_metrics = metrics_by_phase[2]
        performance_metrics = {
            'accuracy': calculate_metric_stats(phase2_metrics, 'accuracy'),
            'f1': calculate_metric_stats(phase2_metrics, 'f1'),
            'precision': calculate_metric_stats(phase2_metrics, 'precision'),
            'recall': calculate_metric_stats(phase2_metrics, 'recall')
        }

        # Calculate sentiment distribution
        sentiment_distribution = {'positive': 0, 'neutral': 0, 'negative': 0}
        if self.all_predictions:
            distribution = (self.all_predictions
                .groupBy('predicted_sentiment')
                .count()
                .collect())
            total_predictions = __builtins__.sum([row['count'] for row in distribution])
            for row in distribution:
                sentiment = row['predicted_sentiment'].lower()
                if sentiment in sentiment_distribution:
                    sentiment_distribution[sentiment] = row['count'] / total_predictions

        return {
            'total_batches': len(self.batch_metrics) // 2,  # Divide by 2 for phase1 and phase2
            'total_reviews': total_reviews,
            'avg_reviews_per_batch': total_reviews / (len(self.batch_metrics) // 2) if self.batch_metrics else 0,
            'performance_metrics': performance_metrics,
            'sentiment_distribution': sentiment_distribution,
            'batch_trend': [
                {
                    'batch_id': m['batch_id'],
                    'accuracy': m['accuracy'],
                    'f1': m['f1']
                } for m in phase2_metrics if 'batch_id' in m
            ]
        }

    def visualize_results(self):
        """Create comprehensive visualizations of batch processing results"""
        stats = self.get_batch_statistics()
    
        # Set up a larger figure to accommodate all plots
        plt.figure(figsize=(20, 15))
    
        # Plot 1: Performance Metrics (unchanged)
        plt.subplot(2, 2, 1)
        metrics = stats['performance_metrics']
        metric_names = list(metrics.keys())
        mean_values = [metrics[m]['mean'] for m in metric_names]
    
        bars = plt.bar(metric_names, mean_values)
        plt.title('Average Performance Metrics', fontsize=12, pad=20)
        plt.ylabel('Score')
        plt.ylim(0, 1)
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.3f}', ha='center', va='bottom')
    
        # Plot 2: Sentiment Distribution (unchanged)
        plt.subplot(2, 2, 2)
        sentiment_dist = stats['sentiment_distribution']
        colors = ['#2ecc71', '#95a5a6', '#e74c3c']  # Green, Gray, Red
        plt.pie(sentiment_dist.values(), labels=sentiment_dist.keys(),
                autopct='%1.1f%%', colors=colors, startangle=90)
        plt.title('Overall Sentiment Distribution', fontsize=12, pad=20)
    
        # Sort batch_trend by batch_id for chronological order
        batch_trend = sorted(stats['batch_trend'], key=lambda x: x['batch_id'])
    
        # Plot 3: Accuracy Trend Across Batches
        plt.subplot(2, 2, 3)
        if batch_trend:
            batch_ids = [b['batch_id'] for b in batch_trend]
            accuracies = [b['accuracy'] for b in batch_trend]
    
            plt.plot(batch_ids, accuracies, marker='o', linewidth=2, markersize=8)
            plt.title('Accuracy Trend Across Batches', fontsize=12, pad=20)
            plt.xticks(rotation=45, ha='right')  # Rotate labels and align right to avoid overlap
            plt.ylabel('Accuracy')
            plt.grid(True, linestyle='--', alpha=0.7)  # Add a subtle grid
            # Add value labels above each point
            for i, v in enumerate(accuracies):
                plt.text(i, v, f'{v:.3f}', ha='center', va='bottom')
    
        # Plot 4: F1 Score Trend Across Batches
        plt.subplot(2, 2, 4)
        if batch_trend:
            batch_ids = [b['batch_id'] for b in batch_trend]
            f1_scores = [b['f1'] for b in batch_trend]
    
            plt.plot(batch_ids, f1_scores, marker='o', color='orange',
                     linewidth=2, markersize=8)
            plt.title('F1 Score Trend Across Batches', fontsize=12, pad=20)
            plt.xticks(rotation=45, ha='right')  # Rotate labels and align right
            plt.ylabel('F1 Score')
            plt.grid(True, linestyle='--', alpha=0.7)  # Add a subtle grid
            # Add value labels above each point
            for i, v in enumerate(f1_scores):
                plt.text(i, v, f'{v:.3f}', ha='center', va='bottom')
    
        # Adjust layout to prevent overlap
        plt.tight_layout()
        plt.show()
    
        return stats

In [ ]:
# Cell 16: Main Execution
def run_batch_processing(spark, file_path, time_windows):
    """Main function to run the batch processing with improved error handling"""
    try:
        # Initialize components
        analyzer = ReviewAnalyzer(spark)
        sentiment_analyzer = SentimentAnalyzer(spark)

        # Create and run batch processor
        processor = BatchProcessor(spark, file_path, analyzer, sentiment_analyzer)
        final_predictions = processor.process_all_batches(time_windows)

        if final_predictions:
            # Get and display statistics
            stats = processor.get_batch_statistics()
            processor.visualize_results()

            print("\nBatch Processing Summary:")
            print(f"Total Batches Processed: {stats['total_batches']}")

            print("\nPerformance Metrics:")
            for metric, values in stats['performance_metrics'].items():
                print(f"{metric.title()}: Mean={values['mean']:.3f}, "
                      f"Min={values['min']:.3f}, Max={values['max']:.3f}")

            print("\nSentiment Distribution:")
            for sentiment, ratio in stats['sentiment_distribution'].items():
                print(f"{sentiment.title()}: {ratio*100:.1f}%")

            # Aggregate final results
            product_sentiments = (
                final_predictions
                .groupBy("ProductId")
                .agg(
                    collect_list("predicted_sentiment").alias("sentiments"),
                    count("*").alias("review_count"),
                    avg(when(col("predicted_sentiment") == "Positive", 1.0).otherwise(0.0))
                        .alias("positive_ratio"),
                    avg(when(col("predicted_sentiment") == "Negative", 1.0).otherwise(0.0))
                        .alias("negative_ratio"),
                    avg(when(col("predicted_sentiment") == "Neutral", 1.0).otherwise(0.0))
                        .alias("neutral_ratio")
                )
                .withColumn(
                    "dominant_sentiment",
                    when(
                        (col("positive_ratio") > col("negative_ratio")) &
                        (col("positive_ratio") > col("neutral_ratio")),
                        "Positive"
                    )
                    .when(
                        (col("negative_ratio") > col("positive_ratio")) &
                        (col("negative_ratio") > col("neutral_ratio")),
                        "Negative"
                    )
                    .otherwise("Neutral")
                )
            )

            return product_sentiments

        else:
            print("No predictions were generated")
            return None

    except Exception as e:
        print(f"Error in batch processing: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
# Cell 17: Execute Analysis

# Get time range from the original preprocessed data
min_max = preprocessed_data.select(
    spark_min(col("RawTimestamp").cast("long")).alias("min_ts"),
    spark_max(col("RawTimestamp").cast("long")).alias("max_ts")
).first()

start_date = datetime.utcfromtimestamp(min_max["min_ts"])
end_date = datetime.utcfromtimestamp(min_max["max_ts"])

# Generate quarterly time windows
current_date = start_date.replace(day=1, hour=0, minute=0, second=0)
time_windows = []
while current_date <= end_date:
    window_start = current_date
    window_end = current_date + relativedelta(months=3)
    time_windows.append((
        int(window_start.timestamp()),
        int(window_end.timestamp())
    ))
    current_date = window_end

# Process data in time windows (this returns predictions, not training data)
final_predictions = run_batch_processing(spark, file_path, time_windows)

# Load full dataset and preprocess it for training
raw_df = spark.read.csv(file_path, header=True, inferSchema=True)
training_data = sentiment_analyzer.preprocess_text(raw_df)

# Train both Phase 1 and Phase 2 models on real review data
sentiment_analyzer.train_phase1_classifier(training_data)
sentiment_analyzer.train_phase2_classifier(training_data)

# Generate predictions on the full dataset
predicted_df = sentiment_analyzer.predict(training_data)

# Check if predictions were made
if predicted_df:
    print(f"Successfully processed {predicted_df.count()} predictions")
    # You can now visualize or aggregate results as needed
else:
    print("No predictions were made. Check for issues in the data or processing logic.")

In [ ]:
#cell 18:
sentiment_data.groupBy("sentiment_polarity").count().show()

In [ ]:
#cell 20:
if 'predicted_df' in globals():
    print("predicted_df is defined.")
    predicted_df.printSchema()
else:
    print("predicted_df is not defined. Please run Cell 17 first.")

In [ ]:
#cell 21:

from pyspark.ml.feature import Tokenizer, NGram, CountVectorizer
from pyspark.sql.functions import udf, col, explode, array_distinct, flatten
from pyspark.sql.types import ArrayType, StringType

def extract_features_with_ngrams(df, text_col="Text", sentiment_col="predicted_sentiment"):
    # Tokenize the review text into words if not already tokenized
    if "words" not in df.columns:
        tokenizer = Tokenizer(inputCol=text_col, outputCol="words")
        df = tokenizer.transform(df)

    # Generate unigrams (1-grams) and bigrams (2-grams)
    unigram = NGram(n=1, inputCol="words", outputCol="unigrams")
    bigram = NGram(n=2, inputCol="words", outputCol="bigrams")
    df = unigram.transform(bigram.transform(df))

    # Use CountVectorizer to filter frequent N-grams
    cv_unigram = CountVectorizer(inputCol="unigrams", outputCol="unigram_features", vocabSize=100, minDF=5.0)
    cv_bigram = CountVectorizer(inputCol="bigrams", outputCol="bigram_features", vocabSize=100, minDF=3.0)

    # Fit and transform to get feature vectors
    cv_unigram_model = cv_unigram.fit(df)
    cv_bigram_model = cv_bigram.fit(df)
    feature_df = cv_unigram_model.transform(cv_bigram_model.transform(df))

    # Extract vocabularies for mapping indices to terms
    unigram_vocab = cv_unigram_model.vocabulary
    bigram_vocab = cv_bigram_model.vocabulary

    # Define UDF to pair present N-grams with sentiments
    @udf(ArrayType(StringType()))
    def extract_present_features(unigram_vec, bigram_vec, sentiment):
        unigram_indices = [int(i) for i, v in zip(unigram_vec.indices, unigram_vec.values) if v > 0]
        bigram_indices = [int(i) for i, v in zip(bigram_vec.indices, bigram_vec.values) if v > 0]

        unigram_pairs = [f"{unigram_vocab[i]}:{sentiment}" for i in unigram_indices if i < len(unigram_vocab)]
        bigram_pairs = [f"{bigram_vocab[i]}:{sentiment}" for i in bigram_indices if i < len(bigram_vocab)]

        return unigram_pairs + bigram_pairs

    # Add feature-sentiment pairs to the DataFrame
    feature_sentiment_df = feature_df.withColumn(
        "feature_sentiment_pairs",
        extract_present_features(col("unigram_features"), col("bigram_features"), col(sentiment_col))
    )

    return feature_sentiment_df

# Apply to predicted_df
feature_sentiment_df = extract_features_with_ngrams(predicted_df)

# Display results
display_df = feature_sentiment_df.select(
    col("Text").substr(1, 50).alias("Text_Shortened"),  # Shorten Text to 50 characters
    "predicted_sentiment",
    col("feature_sentiment_pairs").cast("string").alias("feature_sentiment_pairs")  # Convert array to string
)

# Show the first 5 rows with adjusted column widths
display_df.show(5, truncate=False)

In [ ]:
# Cell 22: Pattern Mining with Progress Tracking
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.functions import col, array_distinct
from itertools import combinations
import time
from datetime import datetime

# Trie implementation (unchanged)
class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0
        self.is_end = False

def insert_into_trie(root, itemset, count):
    node = root
    for item in sorted(itemset):
        if item not in node.children:
            node.children[item] = TrieNode()
        node = node.children[item]
    node.is_end = True
    node.count += count

# Apriori with progress tracking
def apriori_with_trie(transactions_rdd, min_support=0.05, max_iter=3):
    print(f"Starting Apriori at {datetime.now()}")
    start_time = time.time()

    num_transactions = transactions_rdd.count()
    min_count = num_transactions * min_support
    print(f"Number of transactions: {num_transactions}, Min count: {min_count}")

    root = TrieNode()

    # First pass: Find frequent 1-itemsets
    print("Finding frequent 1-itemsets...")
    items = transactions_rdd.flatMap(lambda x: [(item, 1) for item in x]).reduceByKey(lambda a, b: a + b)
    frequent_items = items.filter(lambda x: x[1] >= min_count).collect()
    print(f"Found {len(frequent_items)} frequent 1-itemsets in {time.time() - start_time:.2f} seconds")

    for item, count in frequent_items:
        insert_into_trie(root, (item,), count)

    frequent_itemsets = [(item,) for item, _ in frequent_items]
    k = 2

    # Subsequent passes: Generate and count larger itemsets
    while k <= max_iter:
        print(f"Iteration {k} (k={k}) at {datetime.now()}")
        iter_start = time.time()

        candidates = transactions_rdd.flatMap(
            lambda itemset: [tuple(sorted(combo)) for combo in combinations(itemset, k)]
        ).distinct()
        print(f"Generated {candidates.count()} candidates for k={k} in {time.time() - iter_start:.2f} seconds")

        candidate_counts = candidates.map(lambda c: (c, 1)).reduceByKey(lambda a, b: a + b)
        freq_candidates = candidate_counts.filter(lambda x: x[1] >= min_count).collect()
        print(f"Found {len(freq_candidates)} frequent {k}-itemsets in {time.time() - iter_start:.2f} seconds")

        if not freq_candidates:
            print(f"No frequent {k}-itemsets found, stopping Apriori.")
            break

        for itemset, count in freq_candidates:
            insert_into_trie(root, itemset, count)
            frequent_itemsets.append(itemset)

        k += 1

    total_time = time.time() - start_time
    print(f"Apriori completed in {total_time:.2f} seconds")
    return frequent_itemsets

# FP-Growth with progress tracking
def run_pattern_mining_enhanced(product_sentiments_df, min_support=0.05, min_confidence=0.3, max_iter=3):
    print(f"Starting pattern mining at {datetime.now()}")

    # Prepare transactions for Apriori
    print("Preparing transactions for Apriori...")
    start_time = time.time()
    transactions_rdd = product_sentiments_df.rdd.map(
        lambda row: tuple(sorted(set(row["sentiments"]))) if row["sentiments"] else tuple()
    ).repartition(4)  # Match CPU cores
    print(f"Prepared transactions in {time.time() - start_time:.2f} seconds")

    # Run Apriori
    apriori_results = apriori_with_trie(transactions_rdd, min_support, max_iter)

    # Run FP-Growth
    print(f"Starting FP-Growth at {datetime.now()}")
    start_time = time.time()
    fp_growth = FPGrowth(itemsCol="sentiments", minSupport=min_support, minConfidence=min_confidence)
    prepared_df = product_sentiments_df.select(
        "ProductId", array_distinct(col("sentiments")).alias("sentiments")
    ).repartition(4)
    print(f"Prepared DataFrame for FP-Growth in {time.time() - start_time:.2f} seconds")

    fp_model = fp_growth.fit(prepared_df)
    print(f"FP-Growth completed in {time.time() - start_time:.2f} seconds")

    return apriori_results, fp_model.freqItemsets, fp_model.associationRules

# Integrate with Cell 20
product_sentiments_df = feature_sentiment_df.select(
    "ProductId", col("feature_sentiment_pairs").alias("sentiments")
)

# Run pattern mining with progress tracking
apriori_results, fp_results, assoc_rules = run_pattern_mining_enhanced(
    product_sentiments_df, min_support=0.05, min_confidence=0.3, max_iter=3
)

# Display results
if fp_results:
    print("Frequent Itemsets from FP-Growth:")
    fp_results.show(5,truncate=False)
if assoc_rules:
    print("Association Rules from FP-Growth:")
    assoc_rules.show(5,truncate=False)
print(f"Apriori Results: {len(apriori_results)} frequent itemsets found")

In [ ]:
# Cell 23: Visualizations with Corrected Variables
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql.functions import col, from_unixtime, year, count, sum, when

# Define visualization functions (unchanged)
def visualize_sentiment_distribution(sentiment_data):
    if sentiment_data is None or sentiment_data.count() == 0:
        print("Error: Sentiment data is empty or undefined.")
        return
    sentiment_counts = sentiment_data.groupBy("predicted_sentiment").count().toPandas()
    # Convert inf to NaN
    sentiment_counts.replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    sentiment_counts.dropna(inplace=True)  # Optional: Drop NaN if needed
    plt.figure(figsize=(8, 5))
    sns.barplot(x="count", y="predicted_sentiment", data=sentiment_counts)
    plt.title("Sentiment Distribution")
    plt.xlabel("Count")
    plt.ylabel("Sentiment")
    plt.show()

def visualize_top_products_by_sentiment(final_predictions, top_n=10):
    """Visualize top products by positive and negative sentiment ratios."""
    if final_predictions is None or 'positive_ratio' not in final_predictions.columns:
        print("Error: Final predictions data is invalid or missing required columns.")
        return
    top_positive = final_predictions.orderBy(col("positive_ratio").desc()).limit(top_n).toPandas()
    top_negative = final_predictions.orderBy(col("negative_ratio").desc()).limit(top_n).toPandas()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    sns.barplot(x="positive_ratio", y="ProductId", data=top_positive, ax=ax1, color='green')
    ax1.set_title(f"Top {top_n} Products by Positive Sentiment")
    sns.barplot(x="negative_ratio", y="ProductId", data=top_negative, ax=ax2, color='red')
    ax2.set_title(f"Top {top_n} Products by Negative Sentiment")
    plt.tight_layout()
    plt.show()

def visualize_frequent_patterns(frequent_itemsets_fp):
    """Visualize the top frequent itemsets."""
    if frequent_itemsets_fp is not None and frequent_itemsets_fp.count() > 0:
        itemsets_pd = frequent_itemsets_fp.toPandas()
        itemsets_pd['items_str'] = itemsets_pd['items'].apply(lambda x: ', '.join(x))
        top_itemsets = itemsets_pd.nlargest(10, 'freq')
        plt.figure(figsize=(10, 6))
        plt.barh(top_itemsets['items_str'], top_itemsets['freq'], color='skyblue')
        plt.title('Top 10 Frequent Itemsets')
        plt.xlabel('Frequency')
        plt.ylabel('Itemsets')
        plt.show()

def visualize_association_rules(association_rules, min_confidence=0.5, top_n=10):
    """Visualize the top association rules by confidence."""
    if association_rules is not None and association_rules.count() > 0:
        rules_pd = association_rules.toPandas()
        filtered_rules = rules_pd[rules_pd['confidence'] >= min_confidence].nlargest(top_n, 'confidence')
        filtered_rules['rule_str'] = filtered_rules.apply(
            lambda x: f"{' + '.join(x['antecedent'])} → {' + '.join(x['consequent'])}", axis=1
        )
        plt.figure(figsize=(10, 6))
        plt.barh(filtered_rules['rule_str'], filtered_rules['confidence'], color='orange')
        plt.title(f'Top {top_n} Association Rules by Confidence')
        plt.xlabel('Confidence')
        plt.ylabel('Rules')
        plt.show()

def visualize_sentiment_patterns_over_time(time_patterns):
    """Visualize the number of frequent patterns over time."""
    if time_patterns is not None and len(time_patterns) > 0:
        years = sorted(time_patterns.keys())
        pattern_counts = [
            time_patterns[year].count() if time_patterns[year] is not None else 0 for year in years
        ]
        plt.figure(figsize=(10, 6))
        plt.bar(years, pattern_counts, color='navy')
        plt.title('Total Frequent Patterns by Year')
        plt.xlabel('Year')
        plt.ylabel('Pattern Count')
        plt.show()

# Generate time_patterns with null filtering
joined_df = predicted_df.select("ProductId", "Time").join(
    feature_sentiment_df.select("ProductId", "feature_sentiment_pairs"), "ProductId"
)
joined_df = joined_df.withColumn("year", year(from_unixtime(col("Time"))))
joined_df = joined_df.filter(col("year").isNotNull())  # Filter out null years
time_patterns = {}
distinct_years = joined_df.select("year").distinct().collect()
for yr in distinct_years:
    year_value = yr["year"]
    patterns_for_year = joined_df.filter(col("year") == year_value).select("feature_sentiment_pairs")
    time_patterns[year_value] = patterns_for_year

# Compute final_predictions with review count
final_predictions = predicted_df.groupBy("ProductId").agg(
    (sum(when(col("predicted_sentiment") == "Positive", 1).otherwise(0)) / count("*")).alias("positive_ratio"),
    (sum(when(col("predicted_sentiment") == "Negative", 1).otherwise(0)) / count("*")).alias("negative_ratio"),
    count("*").alias("review_count")
)
final_predictions_filtered = final_predictions.filter(col("review_count") > 1)

# Call visualizations
visualize_sentiment_distribution(predicted_df)
visualize_top_products_by_sentiment(final_predictions_filtered)
visualize_frequent_patterns(fp_results)
visualize_association_rules(assoc_rules)
visualize_sentiment_patterns_over_time(time_patterns)

In [ ]:
# Cell 24: Enhanced Pattern Mining and Visualization with Fixed Warnings
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.functions import (
    col, array_distinct, from_unixtime, year, explode, collect_list,
    count, sum, when  # Added these imports to match cell 23
)
from itertools import combinations
import time
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings

# Suppress FutureWarnings as a fallback
warnings.filterwarnings("ignore", category=FutureWarning)

# Define Trie for Apriori (from Cell 22)
class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0
        self.is_end = False

def insert_into_trie(root, itemset, count):
    node = root
    for item in sorted(itemset):
        if item not in node.children:
            node.children[item] = TrieNode()
        node = node.children[item]
    node.is_end = True
    node.count += count

# Apriori with progress tracking (from Cell 22)
def apriori_with_trie(transactions_rdd, min_support=0.05, max_iter=3):
    print(f"Starting Apriori at {datetime.now()}")
    start_time = time.time()

    num_transactions = transactions_rdd.count()
    min_count = num_transactions * min_support
    print(f"Number of transactions: {num_transactions}, Min count: {min_count}")

    root = TrieNode()

    # First pass: Find frequent 1-itemsets
    print("Finding frequent 1-itemsets...")
    items = transactions_rdd.flatMap(lambda x: [(item, 1) for item in x]).reduceByKey(lambda a, b: a + b)
    frequent_items = items.filter(lambda x: x[1] >= min_count).collect()
    print(f"Found {len(frequent_items)} frequent 1-itemsets in {time.time() - start_time:.2f} seconds")

    for item, count in frequent_items:
        insert_into_trie(root, (item,), count)

    frequent_itemsets = [(item,) for item, _ in frequent_items]
    k = 2

    # Subsequent passes: Generate and count larger itemsets
    while k <= max_iter:
        print(f"Iteration {k} (k={k}) at {datetime.now()}")
        iter_start = time.time()

        candidates = transactions_rdd.flatMap(
            lambda itemset: [tuple(sorted(combo)) for combo in combinations(itemset, k)]
        ).distinct()
        print(f"Generated {candidates.count()} candidates for k={k} in {time.time() - iter_start:.2f} seconds")

        candidate_counts = candidates.map(lambda c: (c, 1)).reduceByKey(lambda a, b: a + b)
        freq_candidates = candidate_counts.filter(lambda x: x[1] >= min_count).collect()
        print(f"Found {len(freq_candidates)} frequent {k}-itemsets in {time.time() - iter_start:.2f} seconds")

        if not freq_candidates:
            print(f"No frequent {k}-itemsets found, stopping Apriori.")
            break

        for itemset, count in freq_candidates:
            insert_into_trie(root, itemset, count)
            frequent_itemsets.append(itemset)

        k += 1

    total_time = time.time() - start_time
    print(f"Apriori completed in {total_time:.2f} seconds")
    return frequent_itemsets, total_time

# Enhanced pattern mining with time-based patterns
def enhanced_pattern_mining(predictions_df, sentiments_df, min_support=0.05):
    print(f"Starting enhanced_pattern_mining at {datetime.now()}")
    start_time = time.time()

    # Run FP-Growth
    fp = FPGrowth(itemsCol="sentiments", minSupport=min_support, minConfidence=0.5)
    model = fp.fit(sentiments_df)
    feature_patterns = model.freqItemsets
    sentiment_patterns = model.freqItemsets
    sentiment_rules = model.associationRules

    # Generate time_patterns
    joined_df = predictions_df.select("ProductId", "Time").join(
        sentiments_df.select("ProductId", "sentiments"),
        "ProductId"
    )
    joined_df = joined_df.withColumn("year", year(from_unixtime(col("Time"))))

    time_patterns = {}
    distinct_years = joined_df.select("year").distinct().filter(col("year").isNotNull()).collect()
    for yr in distinct_years:
        year_value = yr["year"]
        year_data = joined_df.filter(col("year") == year_value)
        # Fixed: Use 'sentiments' instead of 'feature_sentiment_pairs'
        patterns_for_year = year_data.select("sentiments")
        time_patterns[year_value] = patterns_for_year

    total_time = time.time() - start_time
    print(f"enhanced_pattern_mining completed in {total_time:.2f} seconds")
    return feature_patterns, sentiment_patterns, sentiment_rules, time_patterns, total_time

# Run pattern mining with Apriori and FP-Growth (from Cell 22)
def run_pattern_mining_enhanced(sentiments_df, min_support=0.05, min_confidence=0.3, max_iter=3):
    print(f"Starting pattern mining at {datetime.now()}")

    # Prepare transactions for Apriori
    print("Preparing transactions for Apriori...")
    start_time = time.time()
    transactions_rdd = sentiments_df.rdd.map(
        lambda row: tuple(sorted(set(row["sentiments"]))) if row["sentiments"] else tuple()
    ).repartition(4)
    print(f"Prepared transactions in {time.time() - start_time:.2f} seconds")

    # Run Apriori
    apriori_results, apriori_time = apriori_with_trie(transactions_rdd, min_support, max_iter)

    # Run FP-Growth
    print(f"Starting FP-Growth at {datetime.now()}")
    start_time = time.time()
    fp_growth = FPGrowth(itemsCol="sentiments", minSupport=min_support, minConfidence=min_confidence)
    prepared_df = sentiments_df.select(
        "ProductId", array_distinct(col("sentiments")).alias("sentiments")
    ).repartition(4)
    print(f"Prepared DataFrame for FP-Growth in {time.time() - start_time:.2f} seconds")

    fp_model = fp_growth.fit(prepared_df)
    fp_time = time.time() - start_time
    print(f"FP-Growth completed in {fp_time:.2f} seconds")

    return apriori_results, fp_model.freqItemsets, fp_model.associationRules, apriori_time, fp_time

# Updated Visualization Functions with Fixes
def visualize_sentiment_distribution(sentiment_data):
    if sentiment_data is None or sentiment_data.count() == 0:
        print("Error: Sentiment data is empty or undefined.")
        return
    sentiment_counts = sentiment_data.groupBy("predicted_sentiment").count().toPandas()
    sentiment_counts.replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    sentiment_counts.dropna(inplace=True)
    plt.figure(figsize=(8, 5))
    sns.barplot(x="count", y="predicted_sentiment", data=sentiment_counts)
    plt.title("Sentiment Distribution")
    plt.xlabel("Count")
    plt.ylabel("Sentiment")
    plt.show()

def visualize_top_products_by_sentiment(final_predictions, top_n=10):
    if final_predictions is None or 'positive_ratio' not in final_predictions.columns:
        print("Error: Final predictions data is invalid or missing required columns.")
        return
    top_positive = final_predictions.orderBy(col("positive_ratio").desc()).limit(top_n).toPandas()
    top_negative = final_predictions.orderBy(col("negative_ratio").desc()).limit(top_n).toPandas()
    top_positive.replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    top_negative.replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    top_positive.dropna(inplace=True)
    top_negative.dropna(inplace=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    sns.barplot(x="positive_ratio", y="ProductId", data=top_positive, ax=ax1, color='green')
    ax1.set_title(f"Top {top_n} Products by Positive Sentiment")
    sns.barplot(x="negative_ratio", y="ProductId", data=top_negative, ax=ax2, color='red')
    ax2.set_title(f"Top {top_n} Products by Negative Sentiment")
    plt.tight_layout()
    plt.show()

def visualize_frequent_patterns(frequent_itemsets_fp):
    if frequent_itemsets_fp is not None and frequent_itemsets_fp.count() > 0:
        itemsets_pd = frequent_itemsets_fp.toPandas()
        itemsets_pd['items_str'] = itemsets_pd['items'].apply(lambda x: ', '.join(x))
        top_itemsets = itemsets_pd.nlargest(10, 'freq')
        plt.figure(figsize=(10, 6))
        plt.barh(top_itemsets['items_str'], top_itemsets['freq'], color='skyblue')
        plt.title('Top 10 Frequent Itemsets (FP-Growth)')
        plt.xlabel('Frequency')
        plt.ylabel('Itemsets')
        plt.show()

def visualize_apriori_frequent_patterns(apriori_results, top_n=10):
    if apriori_results is not None and len(apriori_results) > 0:
        apriori_df = pd.DataFrame(apriori_results, columns=['items'])
        apriori_df['items_str'] = apriori_df['items'].apply(lambda x: ', '.join(x))
        apriori_counts = apriori_df['items_str'].value_counts().reset_index()
        apriori_counts.columns = ['items_str', 'freq']
        top_itemsets = apriori_counts.nlargest(top_n, 'freq')
        plt.figure(figsize=(10, 6))
        plt.barh(top_itemsets['items_str'], top_itemsets['freq'], color='purple')
        plt.title(f'Top {top_n} Frequent Itemsets (Apriori)')
        plt.xlabel('Frequency')
        plt.ylabel('Itemsets')
        plt.show()

def visualize_apriori_vs_fpgrowth(apriori_results, fp_results):
    apriori_count = len(apriori_results) if apriori_results is not None else 0
    fp_count = fp_results.count() if fp_results is not None else 0
    comparison_df = pd.DataFrame({
        'Algorithm': ['Apriori', 'FP-Growth'],
        'Itemset Count': [apriori_count, fp_count]
    })
    comparison_df.replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    comparison_df.dropna(inplace=True)
    plt.figure(figsize=(8, 5))
    sns.barplot(x='Itemset Count', y='Algorithm', data=comparison_df, palette='Set2')
    plt.title('Comparison of Frequent Itemsets: Apriori vs FP-Growth')
    plt.xlabel('Number of Frequent Itemsets')
    plt.ylabel('Algorithm')
    plt.show()

def visualize_performance_comparison(apriori_time, fp_time):
    performance_df = pd.DataFrame({
        'Algorithm': ['Apriori', 'FP-Growth'],
        'Runtime (seconds)': [apriori_time, fp_time]
    })
    performance_df.replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    performance_df.dropna(inplace=True)
    plt.figure(figsize=(8, 5))
    sns.barplot(x='Runtime (seconds)', y='Algorithm', data=performance_df, palette='Set1')
    plt.title('Runtime Comparison: Apriori vs FP-Growth')
    plt.xlabel('Runtime (seconds)')
    plt.ylabel('Algorithm')
    plt.show()

def visualize_sentiment_over_time(sentiment_data):
    if sentiment_data is None or sentiment_data.count() == 0:
        print("Error: Sentiment data is empty or undefined.")
        return
    sentiment_data = sentiment_data.withColumn("year", year(from_unixtime(col("Time"))))
    sentiment_by_year = sentiment_data.groupBy("year", "predicted_sentiment").count().toPandas()
    sentiment_by_year.replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    sentiment_by_year.dropna(inplace=True)
    # Pivot to avoid Seaborn grouping issues
    pivot_df = sentiment_by_year.pivot(index='year', columns='predicted_sentiment', values='count').fillna(0)
    pivot_df.plot(kind='line', marker='o', figsize=(10, 6))
    plt.title('Sentiment Distribution Over Time')
    plt.xlabel('Year')
    plt.ylabel('Count')
    plt.legend(title='Sentiment')
    plt.show()

def visualize_association_rules(association_rules, min_confidence=0.5, top_n=10):
    if association_rules is not None and association_rules.count() > 0:
        rules_pd = association_rules.toPandas()
        filtered_rules = rules_pd[rules_pd['confidence'] >= min_confidence].nlargest(top_n, 'confidence')
        filtered_rules['rule_str'] = filtered_rules.apply(
            lambda x: f"{' + '.join(x['antecedent'])} → {' + '.join(x['consequent'])}", axis=1
        )
        plt.figure(figsize=(10, 6))
        plt.barh(filtered_rules['rule_str'], filtered_rules['confidence'], color='orange')
        plt.title(f'Top {top_n} Association Rules by Confidence')
        plt.xlabel('Confidence')
        plt.ylabel('Rules')
        plt.show()

def visualize_association_rules_metrics(association_rules, min_confidence=0.5, top_n=10):
    if association_rules is not None and association_rules.count() > 0:
        rules_pd = association_rules.toPandas()
        filtered_rules = rules_pd[rules_pd['confidence'] >= min_confidence].nlargest(top_n, 'confidence')
        filtered_rules['rule_str'] = filtered_rules.apply(
            lambda x: f"{' + '.join(x['antecedent'])} → {' + '.join(x['consequent'])}", axis=1
        )
        fig, ax = plt.subplots(figsize=(12, 6))
        bar_width = 0.25
        index = range(len(filtered_rules))
        plt.bar([i - bar_width for i in index], filtered_rules['confidence'], bar_width, label='Confidence', color='orange')
        plt.bar(index, filtered_rules['support'], bar_width, label='Support', color='blue')
        plt.bar([i + bar_width for i in index], filtered_rules['lift'], bar_width, label='Lift', color='green')
        plt.title(f'Top {top_n} Association Rules Metrics')
        plt.xlabel('Rules')
        plt.ylabel('Value')
        plt.xticks(index, filtered_rules['rule_str'], rotation=45, ha='right')
        plt.legend()
        plt.tight_layout()
        plt.show()

def visualize_sentiment_patterns_over_time(time_patterns):
    if time_patterns is not None and len(time_patterns) > 0:
        years = sorted(time_patterns.keys())
        pattern_counts = [
            time_patterns[year].count() if time_patterns[year] is not None else 0 for year in years
        ]
        plt.figure(figsize=(10, 6))
        plt.bar(years, pattern_counts, color='navy')
        plt.title('Total Frequent Patterns by Year')
        plt.xlabel('Year')
        plt.ylabel('Pattern Count')
        plt.show()

# Compute final_predictions with review count using the correct imports
final_predictions = predicted_df.groupBy("ProductId").agg(
    (sum(when(col("predicted_sentiment") == "Positive", 1).otherwise(0)) / count("*")).alias("positive_ratio"),
    (sum(when(col("predicted_sentiment") == "Negative", 1).otherwise(0)) / count("*")).alias("negative_ratio"),
    count("*").alias("review_count")
)
final_predictions_filtered = final_predictions.filter(col("review_count") > 1)

# Run enhanced pattern mining
feature_patterns, sentiment_patterns, sentiment_rules, time_patterns, enhanced_time = enhanced_pattern_mining(
    predicted_df, product_sentiments_df, min_support=0.05
)

apriori_results, fp_results, assoc_rules, apriori_time, fp_time = run_pattern_mining_enhanced(
    product_sentiments_df, min_support=0.05
)

# Generate visualizations with all results
# Classification Visualizations
visualize_sentiment_distribution(predicted_df)  # Sentiment distribution
visualize_top_products_by_sentiment(final_predictions_filtered)  # Top products by sentiment, using filtered version
visualize_sentiment_over_time(predicted_df)  # Sentiment over time

# Pattern Mining Visualizations
visualize_frequent_patterns(fp_results)  # FP-Growth frequent itemsets
visualize_apriori_frequent_patterns(apriori_results)  # Apriori frequent itemsets
visualize_apriori_vs_fpgrowth(apriori_results, fp_results)  # Compare Apriori vs FP-Growth itemset counts
visualize_performance_comparison(apriori_time, fp_time)  # Compare runtime
visualize_association_rules(assoc_rules)  # Association rules by confidence
visualize_association_rules_metrics(assoc_rules)  # Association rules with confidence, support, lift
visualize_sentiment_patterns_over_time(time_patterns)  # Patterns over time